# 🔗 Entity Linking — Synthetic Dataset Generation

Pipeline : sous-graphe Wikidata → génération de textes avec **mentions naturellement imprécises** → dataset bi-encodeur pour l'entraînement

**Principe :**
- Qwen génère des textes où les entités sont mentionnées de façon variée (nom partiel, alias, paraphrase)
- On extrait les spans de mentions + leur entité cible (QID)
- On construit des paires `(mention_in_context, entity_description)` avec labels `1` (correct) / `0` (négatif difficile)

---
1. Setup
2. Génération de textes avec mentions variées
3. Pipeline complet
4. Construction des paires bi-encodeur
5. Visualisation & stats
6. Export

## 1. Setup

In [ ]:
!pip install -q openai pandas tqdm

In [ ]:
import os, json, re, time, random
from collections import defaultdict
from openai import OpenAI
from tqdm import tqdm
import pandas as pd
from IPython.display import display, HTML

# ── Alibaba DashScope ─────────────────────────────────────────────────────────
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY", "sk-460e71b3918840fb9bbc8b438541dd11")
MODEL             = "qwen-plus"

client = OpenAI(
    api_key  = DASHSCOPE_API_KEY,
    base_url = "https://dashscope-us.aliyuncs.com/compatible-mode/v1",
)

def call_llm(messages, temperature=0.7, max_tokens=800):
    t0 = time.time()
    response = client.chat.completions.create(
        model       = MODEL,
        messages    = messages,
        temperature = temperature,
        max_tokens  = max_tokens,
    )
    return {
        "text":       response.choices[0].message.content,
        "latency_ms": int((time.time() - t0) * 1000),
        "tokens":     response.usage.total_tokens,
    }

r = call_llm([{"role": "user", "content": "Reply: OK"}])
print(f"✅ Qwen API OK — {r['latency_ms']} ms")

✅ Qwen API OK — 3301 ms


In [ ]:
# ── Chargement du sous-graphe ─────────────────────────────────────────────────
SUBGRAPH_CSV = "subgraph.csv"

RELATION_EN = {
    "réalisateur":         "director",
    "acteur":              "cast member",
    "genre":               "genre",
    "date de publication": "publication date",
    "pays d'origine":      "country of origin",
}
PROP_TYPES = {
    "P57":  "person",
    "P161": "person",
    "P136": "genre",
    "P577": "date",
    "P495": "country",
}

df = pd.read_csv(SUBGRAPH_CSV)
df["relation_en"] = df["prop_label"].map(RELATION_EN)

# Grouper les triplets par film
film_groups = defaultdict(list)
for _, row in df.iterrows():
    film_groups[row["sujet_id"]].append({
        "subject":   {"id": row["sujet_id"],  "label": row["sujet_label"], "type": "film"},
        "predicate": {"id": row["prop_id"],   "label": row["relation_en"]},
        "object":    {"id": row["objet_id"],  "label": row["objet_label"],
                      "type": PROP_TYPES.get(row["prop_id"], "entity")},
    })

# Catalogue d'entités (pour les négatifs)
ENTITY_CATALOG = {}
for _, row in df.iterrows():
    ENTITY_CATALOG[row["sujet_id"]] = {
        "id": row["sujet_id"], "label": row["sujet_label"], "type": "film",
        "description": f"film: {row['sujet_label']}"
    }
    ENTITY_CATALOG[row["objet_id"]] = {
        "id": row["objet_id"], "label": row["objet_label"],
        "type": PROP_TYPES.get(row["prop_id"], "entity"),
        "description": f"{PROP_TYPES.get(row['prop_id'], 'entity')}: {row['objet_label']}"
    }

# Filtrer : films avec >= 2 triplets
film_groups = {k: v for k, v in film_groups.items() if len(v) >= 2}
print(f"✅ {len(film_groups)} films | {len(ENTITY_CATALOG)} entités dans le catalogue")

✅ 138 films | 859 entités dans le catalogue


## 2. Prompt engineering

Le prompt demande à Qwen de :
1. Écrire un paragraphe naturel à partir des triplets
2. **Utiliser des mentions imprécises** (prénom seul, surnom, paraphrase) pour au moins la moitié des entités
3. Retourner un JSON structuré avec les spans exacts de chaque mention et un flag `is_partial`

In [ ]:
SYSTEM_EL = """You are an expert at generating training data for Entity Linking.

Given a list of (subject, relation, object) triplets about a film, you must:
1. Write a natural English paragraph (2-4 sentences) that expresses all the facts.
2. Use VARIED and IMPRECISE mentions of entities — like a real journalist or writer would:
   - Use only the first name OR last name instead of the full name
   - Use a pronoun reference after first mention (he, she, it, they)
   - Use a descriptive paraphrase (e.g. "the French director" instead of the name)
   - Abbreviate country or organization names (e.g. "US" for "United States")
   - At least HALF the entities should have imprecise/partial mentions
3. Return a JSON with the paragraph and every mention mapped to its Wikidata entity ID.

Output format (strict JSON, nothing else):
{
  "paragraph": "<the generated text>",
  "mentions": [
    {
      "mention": "<exact surface form as it appears in the paragraph>",
      "entity_id": "<Wikidata QID>",
      "entity_label": "<full canonical label>",
      "entity_type": "<film|person|country|genre|date>",
      "start": <char offset start>,
      "end": <char offset end>,
      "is_partial": <true if mention is imprecise or partial, false if exact match>
    }
  ]
}"""


def build_el_prompt(triplets, max_triplets=5):
    selected   = triplets[:max_triplets]
    film_label = selected[0]["subject"]["label"]
    film_id    = selected[0]["subject"]["id"]

    lines = [
        f'  ({t["subject"]["label"]} [{t["subject"]["id"]}], '
        f'{t["predicate"]["label"]}, '
        f'{t["object"]["label"]} [{t["object"]["id"]}])'
        for t in selected
    ]

    seen, entity_lines = set(), []
    for t in selected:
        for role in ("subject", "object"):
            e = t[role]
            if e["id"] not in seen:
                seen.add(e["id"])
                entity_lines.append(
                    f'  - "{e["label"]}" (ID: {e["id"]}, type: {e.get("type","entity")})'
                )

    return (
        f'Generate an entity-linking training example for the film "{film_label}" ({film_id}).\n'
        f'\nTriplets:\n' + "\n".join(lines) +
        f'\n\nEntities to mention (vary at least half of them):\n'
        + "\n".join(entity_lines) +
        f'\n\nExamples of variation: use "Nolan" instead of "Christopher Nolan", '
        f'"the director" instead of the name, "US" instead of "United States".\n'
        f'Return strict JSON only.'
    )


def parse_el_output(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*",     "", text)
    text = re.sub(r"\s*```$",     "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
    return None


def verify_spans(paragraph, mentions):
    """Vérifie et corrige les offsets retournés par le LLM."""
    verified = []
    for m in mentions:
        surface = m.get("mention", "").strip(".,!?;:\'\"")
        if not surface:
            continue
        idx = paragraph.lower().find(surface.lower())
        if idx != -1:
            m["start"]   = idx
            m["end"]     = idx + len(surface)
            m["mention"] = paragraph[idx: idx + len(surface)]
            verified.append(m)
    return verified


print("✅ Prompts définis")
print("\nExemple de prompt:")
ex = list(film_groups.values())[0]
print(build_el_prompt(ex))

✅ Prompts définis

Exemple de prompt:
Generate an entity-linking training example for the film "Les Lyonnais" (Q593).

Triplets:
  (Les Lyonnais [Q593], director, Olivier Marchal [Q694259])
  (Les Lyonnais [Q593], cast member, Étienne Chicot [Q289524])
  (Les Lyonnais [Q593], cast member, Tchéky Karyo [Q317704])
  (Les Lyonnais [Q593], cast member, François Levantal [Q658664])
  (Les Lyonnais [Q593], cast member, Daniel Duval [Q778573])

Entities to mention (vary at least half of them):
  - "Les Lyonnais" (ID: Q593, type: film)
  - "Olivier Marchal" (ID: Q694259, type: person)
  - "Étienne Chicot" (ID: Q289524, type: person)
  - "Tchéky Karyo" (ID: Q317704, type: person)
  - "François Levantal" (ID: Q658664, type: person)
  - "Daniel Duval" (ID: Q778573, type: person)

Examples of variation: use "Nolan" instead of "Christopher Nolan", "the director" instead of the name, "US" instead of "United States".
Return strict JSON only.


In [ ]:
# ── Test sur un film ─────────────────────────────────────────────────────────
test_triples = list(film_groups.values())[0]
film_name    = test_triples[0]["subject"]["label"]

print(f"🎬 Film: {film_name}")
for t in test_triples:
    print(f"  [{t['predicate']['label']}] → {t['object']['label']}")

result = call_llm([
    {"role": "system", "content": SYSTEM_EL},
    {"role": "user",   "content": build_el_prompt(test_triples)},
])
parsed = parse_el_output(result["text"])

if parsed:
    paragraph = parsed["paragraph"]
    mentions  = verify_spans(paragraph, parsed.get("mentions", []))
    print(f"\n📝 Paragraphe:\n   {paragraph}")
    print(f"\n🏷  Mentions ({len(mentions)}):")
    for m in mentions:
        flag = "⚠️  PARTIAL" if m.get("is_partial") else "✅ exact"
        print(f"   '{m['mention']}' → {m['entity_label']} ({m['entity_id']}) [{flag}]")
else:
    print("❌ Parse failed. Raw response:")
    print(result["text"][:500])

🎬 Film: Les Lyonnais
  [director] → Olivier Marchal
  [cast member] → Étienne Chicot
  [cast member] → Tchéky Karyo
  [cast member] → François Levantal
  [cast member] → Daniel Duval
  [cast member] → Estelle Skornik
  [cast member] → Francis Renaud
  [cast member] → Gérard Lanvin
  [cast member] → Dimitri Storoge
  [cast member] → Jacques Chambon
  [cast member] → Laurent Olmedo
  [cast member] → Lionnel Astier
  [cast member] → Olivier Chantreau
  [cast member] → Olivier Rabourdin
  [cast member] → Patrick Catalifo
  [cast member] → Simon Astier
  [cast member] → Valeria Cavalli
  [cast member] → Christophe Kourotchkine
  [cast member] → Stéphane Caillard
  [cast member] → Gilles Bénichou
  [genre] → drame
  [genre] → film ou téléfilm de casse
  [genre] → film ou téléfilm criminel
  [publication date] → 2011-11-10T00:00:00Z

📝 Paragraphe:
   The French crime thriller Les Lyonnais features a strong ensemble cast, including Étienne, Tchéky, François, and Daniel. The film was helmed by 

## 3. Pipeline de génération complet

In [ ]:
def generate_el_corpus(film_groups, delay=0.6, max_retries=3, max_films=None):
    corpus, failed = [], []
    film_ids = list(film_groups.keys())
    if max_films:
        film_ids = film_ids[:max_films]

    print(f"🚀 Génération EL — {len(film_ids)} films\n" + "─" * 60)

    for film_id in tqdm(film_ids):
        triplets   = film_groups[film_id]
        film_label = triplets[0]["subject"]["label"]
        success    = False

        for attempt in range(max_retries):
            try:
                result = call_llm([
                    {"role": "system", "content": SYSTEM_EL},
                    {"role": "user",   "content": build_el_prompt(triplets)},
                ])
                parsed = parse_el_output(result["text"])

                if not parsed or "paragraph" not in parsed:
                    raise ValueError("Invalid JSON output")

                paragraph = parsed["paragraph"]
                mentions  = verify_spans(paragraph, parsed.get("mentions", []))

                if len(mentions) == 0:
                    raise ValueError("No valid mentions found")

                corpus.append({
                    "film_id":    film_id,
                    "film_label": film_label,
                    "triplets":   triplets,
                    "paragraph":  paragraph,
                    "mentions":   mentions,
                    "tokens":     result["tokens"],
                    "n_partial":  sum(1 for m in mentions if m.get("is_partial")),
                })
                success = True
                break

            except Exception as e:
                wait = delay * (attempt + 1) * 2
                tqdm.write(f"  [{film_label}] attempt {attempt+1} failed: {e} — waiting {wait:.1f}s")
                time.sleep(wait)

        if not success:
            failed.append(film_label)
        time.sleep(delay)

    print("─" * 60)
    n_mentions = sum(len(r["mentions"]) for r in corpus)
    n_partial  = sum(r["n_partial"] for r in corpus)
    print(f"✅ {len(corpus)} films | {n_mentions} mentions | {n_partial} partielles ({100*n_partial//max(n_mentions,1)}%)")
    if failed:
        print(f"❌ {len(failed)} échecs: {failed[:5]}")
    return corpus


# max_films=None → tous les films ; mettre un entier pour tester
el_corpus = generate_el_corpus(film_groups, delay=0.6, max_films=100)

🚀 Génération EL — 100 films
────────────────────────────────────────────────────────────


 44%|████▍     | 44/100 [10:14<12:33, 13.45s/it]

  [Ghost Rider 2 : L'Esprit de vengeance] attempt 1 failed: Invalid JSON output — waiting 1.2s


 44%|████▍     | 44/100 [10:33<12:33, 13.45s/it]

  [Ghost Rider 2 : L'Esprit de vengeance] attempt 2 failed: Invalid JSON output — waiting 2.4s


...

  [Mama] attempt 2 failed: Invalid JSON output — waiting 2.4s


 60%|██████    | 60/100 [15:48<11:14, 16.85s/it]

  [Mama] attempt 3 failed: Invalid JSON output — waiting 3.6s


100%|██████████| 100/100 [23:30<00:00, 14.10s/it]

────────────────────────────────────────────────────────────
✅ 99 films | 574 mentions | 421 partielles (73%)
❌ 1 échecs: ['Mama']


## 4. Construction des paires bi-encodeur

Pour chaque mention, on construit :
- **1 positif** : `(contexte_avec_mention, description_entité_correcte, label=1)`
- **N négatifs difficiles** : `(contexte_avec_mention, description_entité_même_type, label=0)`

Le contexte utilise des marqueurs `[ENT] mention [/ENT]` pour aider le modèle à localiser la mention.

In [ ]:
CONTEXT_WINDOW    = 64   # caractères autour de la mention
N_HARD_NEGATIVES  = 3    # négatifs par mention


def get_mention_context(paragraph, start, end, window=CONTEXT_WINDOW):
    """
    Extrait le contexte autour d'une mention avec marqueurs spéciaux.
    Format : "...contexte gauche [ENT] mention [/ENT] contexte droit..."
    """
    left    = paragraph[max(0, start - window): start].strip()
    right   = paragraph[end: end + window].strip()
    surface = paragraph[start:end]
    return f"{left} [ENT] {surface} [/ENT] {right}".strip()


def build_entity_description(entity):
    """Description canonique d'une entité pour le bi-encodeur."""
    label = entity.get("label", "")
    etype = entity.get("type",  "entity")
    desc  = entity.get("description", f"{etype}: {label}")
    return f"{label} : {etype} | {desc}"


def sample_hard_negatives(target_id, entity_type, catalog, n=N_HARD_NEGATIVES):
    """Sélectionne des négatifs du même type que la cible (plus difficiles)."""
    same_type = [
        e for eid, e in catalog.items()
        if eid != target_id and e.get("type") == entity_type
    ]
    random.shuffle(same_type)
    negatives = same_type[:n]
    if len(negatives) < n:
        others = [e for eid, e in catalog.items() if eid != target_id and e not in negatives]
        random.shuffle(others)
        negatives += others[:n - len(negatives)]
    return negatives


def build_biencoder_pairs(corpus, catalog, n_hard_neg=N_HARD_NEGATIVES):
    pairs, skipped = [], 0

    for record in corpus:
        paragraph = record["paragraph"]
        for mention in record["mentions"]:
            entity_id   = mention.get("entity_id")
            entity_type = mention.get("entity_type", "entity")
            start       = mention.get("start")
            end         = mention.get("end")
            surface     = mention.get("mention", "")

            if entity_id not in catalog:
                skipped += 1
                continue

            target  = catalog[entity_id]
            context = get_mention_context(paragraph, start, end)
            pos_desc = build_entity_description(target)

            # ── Paire POSITIVE ────────────────────────────────────────────
            pairs.append({
                "mention_context": context,
                "entity_desc":     pos_desc,
                "label":           1,
                "mention_surface": surface,
                "entity_id":       entity_id,
                "entity_label":    target["label"],
                "entity_type":     entity_type,
                "is_partial":      mention.get("is_partial", False),
                "film_id":         record["film_id"],
                "film_label":      record["film_label"],
                "paragraph":       paragraph,
            })

            # ── Paires NÉGATIVES difficiles ───────────────────────────────
            for neg in sample_hard_negatives(entity_id, entity_type, catalog, n_hard_neg):
                pairs.append({
                    "mention_context": context,
                    "entity_desc":     build_entity_description(neg),
                    "label":           0,
                    "mention_surface": surface,
                    "entity_id":       neg["id"],
                    "entity_label":    neg["label"],
                    "entity_type":     neg.get("type", "entity"),
                    "is_partial":      mention.get("is_partial", False),
                    "film_id":         record["film_id"],
                    "film_label":      record["film_label"],
                    "paragraph":       paragraph,
                })

    pos = sum(1 for p in pairs if p["label"] == 1)
    neg = len(pairs) - pos
    partial = sum(1 for p in pairs if p["label"] == 1 and p["is_partial"])
    print(f"✅ {len(pairs)} paires | {pos} positives | {neg} négatives | {skipped} ignorées")
    print(f"   Ratio pos/neg : 1/{neg//max(pos,1)}")
    print(f"   Mentions partielles (positives) : {partial}/{pos} ({100*partial//max(pos,1)}%)")
    return pairs


random.seed(42)
biencoder_pairs = build_biencoder_pairs(el_corpus, ENTITY_CATALOG)

✅ 2288 paires | 572 positives | 1716 négatives | 2 ignorées
   Ratio pos/neg : 1/3
   Mentions partielles (positives) : 419/572 (73%)


## 5. Visualisation & stats

In [ ]:
# ── Aperçu des paires ────────────────────────────────────────────────────────
sample_record = random.choice(el_corpus)
print(f"🎬 Film: {sample_record['film_label']}")
print(f"\n📝 Paragraphe:\n   {sample_record['paragraph']}")
print(f"\n🏷  Mentions annotées:")
for m in sample_record["mentions"]:
    flag = "⚠️  PARTIAL" if m.get("is_partial") else "✅ exact"
    print(f"   [{m['start']}:{m['end']}] '{m['mention']}' → {m['entity_label']} ({m['entity_id']}) {flag}")

print("\n" + "─" * 60 + "\nPaires bi-encodeur pour ce film:")
film_pairs = [p for p in biencoder_pairs if p["film_id"] == sample_record["film_id"]]
for p in film_pairs[:8]:
    sign    = "✅ POS" if p["label"] == 1 else "❌ NEG"
    partial = " [PARTIAL MENTION]" if p["is_partial"] and p["label"] == 1 else ""
    print(f"\n  {sign}{partial}")
    print(f"  mention_context : {p['mention_context']}")
    print(f"  entity_desc     : {p['entity_desc']}")
    print(f"  label           : {p['label']}")

🎬 Film: Twilight, chapitre IV : Révélation, partie 1

📝 Paragraphe:
   The final installment of the vampire saga's second half was helmed by the American filmmaker Bill Condon and released in the US in 2011. Though it's technically the fourth movie in the franchise, many fans refer to it simply as 'Revelation Part 1'. He brought his signature visual style to the adaptation, which originated from Hollywood studios based in the States.

🏷  Mentions annotées:
   [0:55] 'The final installment of the vampire saga's second half' → Twilight, chapitre IV : Révélation, partie 1 (Q60506) ⚠️  PARTIAL
   [93:104] 'Bill Condon' → Bill Condon (Q361336) ✅ exact
   [125:127] 'US' → United States (Q30) ⚠️  PARTIAL
   [328:365] 'Hollywood studios based in the States' → United States (Q30) ⚠️  PARTIAL
   [229:246] 'Revelation Part 1' → Twilight, chapitre IV : Révélation, partie 1 (Q60506) ⚠️  PARTIAL

────────────────────────────────────────────────────────────
Paires bi-encodeur pour ce film:

  ✅ POS [

In [ ]:
# ── Stats globales ───────────────────────────────────────────────────────────
df_pairs = pd.DataFrame(biencoder_pairs)

print("=== Stats du dataset bi-encodeur ===")
print(f"Total paires        : {len(df_pairs)}")
print(f"Positives (label=1) : {(df_pairs.label==1).sum()}")
print(f"Négatives (label=0) : {(df_pairs.label==0).sum()}")
print()
pos_df = df_pairs[df_pairs.label == 1]
print("Distribution par type d'entité (positives):")
print(pos_df["entity_type"].value_counts().to_string())
print()
print("Mentions partielles vs exactes (positives):")
print(pos_df["is_partial"].value_counts().to_string())

=== Stats du dataset bi-encodeur ===
Total paires        : 2288
Positives (label=1) : 572
Négatives (label=0) : 1716

Distribution par type d'entité (positives):
entity_type
person     196
genre      137
film       105
country     78
date        56

Mentions partielles vs exactes (positives):
is_partial
True     419
False    153


In [ ]:
# ── Visualisation HTML avec marquage des mentions ────────────────────────────
COLORS = {
    "film": "#F5C4B3", "person":  "#d4edda",
    "country": "#f8d7da", "genre": "#fff3cd",
    "date": "#e2d9f3", "entity": "#f0f0f0",
}

def render_el_html(record):
    text     = record["paragraph"]
    mentions = sorted(record["mentions"], key=lambda m: m["start"])
    parts, cursor = [], 0
    for m in mentions:
        s, e = m["start"], m["end"]
        if s < cursor: continue
        parts.append(text[cursor:s])
        color   = COLORS.get(m.get("entity_type", "entity"), "#f0f0f0")
        partial = " border:2px dashed #e74c3c;" if m.get("is_partial") else ""
        tooltip = f"{m['entity_label']} ({m['entity_id']}){'  ⚠ partial' if m.get('is_partial') else ''}"
        parts.append(
            f'<mark style="background:{color};border-radius:3px;padding:1px 5px;{partial}" title="{tooltip}">'
            f'{text[s:e]}'
            f'<sup style="font-size:.6em;color:#333">{m["entity_id"]}</sup></mark>'
        )
        cursor = e
    parts.append(text[cursor:])
    return "".join(parts)

html_parts = [
    '<style>div.r{margin:10px 0;padding:14px;border:1px solid #ddd;border-radius:8px;'
    'font-family:sans-serif;line-height:1.9;font-size:14px}</style>'
    '<div style="font-size:11px;color:#888;margin-bottom:8px">'
    '🟩 person &nbsp; 🟥 country &nbsp; 🟧 film &nbsp; 🟨 genre &nbsp; 🟪 date &nbsp;'
    '<span style="border:2px dashed #e74c3c;padding:0 4px">⚠ mention partielle</span></div>'
]
for r in el_corpus[:10]:
    n_partial = sum(1 for m in r["mentions"] if m.get("is_partial"))
    html_parts.append(
        f'<div class="r">'
        f'<div style="font-size:.8em;font-weight:bold;color:#444;margin-bottom:6px">'
        f'🎬 {r["film_label"]} — {len(r["mentions"])} mentions dont {n_partial} partielles</div>'
        f'<div>{render_el_html(r)}</div>'
        f'</div>'
    )
display(HTML("".join(html_parts)))

## 6. Export

In [ ]:
import os
os.makedirs("output", exist_ok=True)

CORPUS_JSONL = "output/el_corpus.jsonl"
PAIRS_JSONL  = "output/el_biencoder_pairs.jsonl"
PAIRS_CSV    = "output/el_biencoder_pairs.csv"

# Corpus brut
with open(CORPUS_JSONL, "w", encoding="utf-8") as f:
    for r in el_corpus:
        out = {
            "film_id":    r["film_id"],
            "film_label": r["film_label"],
            "paragraph":  r["paragraph"],
            "mentions":   r["mentions"],
            "triplets": [
                {"subject": t["subject"]["label"],
                 "relation": t["predicate"]["label"],
                 "object": t["object"]["label"]}
                for t in r["triplets"]
            ],
            "n_partial": r["n_partial"],
            "model":     MODEL,
        }
        f.write(json.dumps(out, ensure_ascii=False) + "\n")

# Paires bi-encodeur JSONL
with open(PAIRS_JSONL, "w", encoding="utf-8") as f:
    for p in biencoder_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# Paires bi-encodeur CSV
df_pairs.to_csv(PAIRS_CSV, index=False, encoding="utf-8")

print(f"✅ Corpus brut  → {CORPUS_JSONL}  ({os.path.getsize(CORPUS_JSONL)//1024} Ko)")
print(f"✅ Paires JSONL → {PAIRS_JSONL}   ({os.path.getsize(PAIRS_JSONL)//1024} Ko)")
print(f"✅ Paires CSV   → {PAIRS_CSV}     ({os.path.getsize(PAIRS_CSV)//1024} Ko)")
print()
print("=== Aperçu final ===")
df_pairs[["mention_surface", "entity_label", "entity_type", "is_partial", "label"]].head(12)

✅ Corpus brut  → output/el_corpus.jsonl  (212 Ko)
✅ Paires JSONL → output/el_biencoder_pairs.jsonl   (1928 Ko)
✅ Paires CSV   → output/el_biencoder_pairs.csv     (1517 Ko)

=== Aperçu final ===


,mention_surface,entity_label,entity_type,is_partial,label
0,Les Lyonnais,Les Lyonnais,film,False,1
1,Les Lyonnais,"Star Wars, épisode III : La Revanche des Sith",film,False,0
2,Les Lyonnais,Harry Potter et la Coupe de feu,film,False,0
3,Les Lyonnais,Le Dernier Maître de l'air,film,False,0
4,Olivier Marchal,Olivier Marchal,person,False,1
5,Olivier Marchal,Navya Nair,person,False,0
6,Olivier Marchal,Albert Hughes,person,False,0
7,Olivier Marchal,Jason Reitman,person,False,0
8,Étienne Chicot,Étienne Chicot,person,False,1
9,Étienne Chicot,Teng Huatao,person,False,0
